# Results Gate Policy

Purpose: tune edge-floor and side-specific policy thresholds, then pick an operating point for live betting.

Use this notebook to answer:
- What edge floor produces stable ROI + CLV?
- Should over/under use different floors?
- Is the gate helping enough to justify stricter filtering?

Edge-floor and recommendation-mix simulator for BET/HOLD policy tuning.

In [4]:
from pathlib import Path
import subprocess
import sys
import polars as pl
from IPython.display import HTML, display

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "production").exists() and (candidate / "src" / "Python").exists():
        ROOT = candidate
        break

sys.path.insert(0, str(ROOT / "src"))
from Python.notebook_analysis_utils import has_over_clv_red_flag

SIM = ROOT / "production" / "ops" / "policy_simulator.py"
OUT_SWEEP = ROOT / "artifacts" / "odds_log" / "policy_scenario_sweep.parquet"


def show_table(df: pl.DataFrame, max_rows: int = 30, height: int = 420):
    pdf = df.to_pandas().round(3)
    if len(pdf) <= max_rows:
        display(pdf)
        return
    table = pdf.to_html(index=False, na_rep="—")
    display(HTML(f"<div style='max-height:{height}px; overflow:auto; border:1px solid #4443; border-radius:6px'>{table}</div>"))


print("repo:", ROOT)
print("simulator:", SIM)

repo: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props
simulator: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\production\ops\policy_simulator.py


In [5]:
thresholds = "0.08,0.10,0.12,0.14,0.16,0.18"
cmd = [sys.executable, str(SIM), "--thresholds", thresholds]
run = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
print(run.stdout)
if run.returncode != 0:
    raise RuntimeError(run.stderr)

--- latest recommendation mix ---
{'recommendation': 'skip', 'oos_reason': None, 'n': 16}
{'recommendation': 'BET', 'oos_reason': None, 'n': 8}
wrote C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\artifacts\odds_log\policy_scenario_sweep.parquet
wrote C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\artifacts\odds_log\policy_scenario_sweep_latest.csv
--- latest scenario rows ---
{'snapshot_utc': '2026-08-12T15:36:20.619632+00:00', 'scope': 'all', 'edge_floor': 0.08, 'n_bets': 222, 'wins': 116, 'losses': 106, 'win_rate': 0.5225225225225225, 'total_pnl': 1652.0471896878437, 'roi': 0.09537124758194322, 'avg_edge': 0.1715317780117607, 'avg_clv_pp': 0.004121526344734585, 'total_stake': 17322.277222686014}
{'snapshot_utc': '2026-08-12T15:36:20.623634+00:00', 'scope': 'all', 'edge_floor': 0.1, 'n_bets': 198, 'wins': 105, 'losses': 93, 'win_rate': 0.5303030303030303, 'total_pnl': 1770.7705572615007, 'roi': 0.10777278611815463, 'avg_edge': 0.18156150561006726, 'avg_clv_pp': 

In [ ]:
if not OUT_SWEEP.exists():
    print("No scenario sweep artifact yet.")
else:
    sweep = pl.read_parquet(OUT_SWEEP)
    latest_ts = sweep.select(pl.col("snapshot_utc").max()).item()
    latest = sweep.filter(pl.col("snapshot_utc") == latest_ts)
    latest_view = latest.sort(["scope", "edge_floor"])
    if "show_table" in globals():
        show_table(latest_view)
    else:
        print(latest_view)

    if "scope" in latest.columns and "n_bets" in latest.columns:
        all_rows = latest.filter(pl.col("scope") == "all")
        if all_rows.height:
            raw_n = int(all_rows.sort("edge_floor").head(1).select("n_bets").item())
            best_n = int(all_rows.sort("edge_floor", descending=True).head(1).select("n_bets").item())
            print(f"raw_n(at lowest floor)={raw_n} best_line_like_n(at highest floor)={best_n}")

    side_view = latest.filter(pl.col("scope").is_in(["over", "under"])) if "scope" in latest.columns else pl.DataFrame()
    if side_view.height:
        side_health = side_view.select([c for c in ["scope", "edge_floor", "roi", "avg_clv_pp", "n_bets"] if c in side_view.columns]).sort(["scope", "edge_floor"])
        print("\nside CLV/ROI health")
        print(side_health)
        side_health_check = side_view.rename({"scope": "side", "avg_clv_pp": "mean_clv_pp"})
        if has_over_clv_red_flag(side_health_check):
            print("RED FLAG: over avg_clv_pp <= 0 for one or more thresholds.")

,snapshot_utc,scope,edge_floor,n_bets,wins,losses,win_rate,total_pnl,roi,avg_edge,avg_clv_pp,total_stake,edge_floor_over,edge_floor_under
0,2026-08-12T15:36:20.691481+00:00,under,0.18,58,34,24,0.586207,1888.508747,0.275139,0.247777,0.009564,6863.83635,NaN,NaN
